In [7]:
import numpy as np

# класс детектор из первой задачи
class Detector:
    def __init__(self, R=2.0, N=40, r=0.1, a=0.5):
        self.R = R # радиус детектора в метрах
        self.N = N # количество ФЭУ
        self.r = r # радиус фотокатода ФЭУ в метрах
        self.a = a # параметр пуассона 

        self.pmt_positions = self.generate_pmt_positions()
        self.pmt_angular_radius = np.arcsin(self.r / self.R)

    # равномерное распределение точек на сфере
    def generate_pmt_positions(self):
        positions = []
        phi = np.pi * (np.sqrt(5.0) - 1.0)  

        for i in range(self.N):
            y = 1.0 - (i / float(self.N - 1)) * 2.0  # y от 1 до -1
            radius_at_y = np.sqrt(1.0 - y * y)
            theta = phi * i

            x = np.cos(theta) * radius_at_y
            z = np.sin(theta) * radius_at_y

            pos = np.array([x, y, z]) * self.R
            positions.append(pos)
            
        return np.array(positions)

    # метод для определения принадлежности точки
    def check_hit_pmt(self, hit_point):
        norm_hit = hit_point / np.linalg.norm(hit_point)
        
        for idx, pmt_pos in enumerate(self.pmt_positions):
            norm_pmt = pmt_pos / self.R
            cos_angle = np.clip(np.dot(norm_hit, norm_pmt), -1.0, 1.0)
            angle = np.arccos(cos_angle)
            
            if angle <= self.pmt_angular_radius:
                return idx # индекс попавшего ФЭУ
                
        return -1


# класс источник из второй задачи
class Source:
    def __init__(self, Y=10000):
        self.Y = Y  # световыход: фотонов / МэВ

    def generate_photons(self, energy, pos=(0.0, 0.0, 0.0)):
        # общее число фотонов пропорционально энергии и световыходу Y
        n_photons = int(energy * self.Y)
        
        # изотропное распределение направлений 
        phi = np.random.uniform(0, 2 * np.pi, n_photons)
        costheta = np.random.uniform(-1.0, 1.0, n_photons)
        sintheta = np.sqrt(1.0 - costheta**2)

        nx = sintheta * np.cos(phi)
        ny = sintheta * np.sin(phi)
        nz = costheta

        directions = np.column_stack((nx, ny, nz))
        positions = np.tile(pos, (n_photons, 1))
        
        return positions, directions


# моделирование распространения и детектирования света
def simulate_single_event(detector, source, energy, source_pos=(0.0, 0.0, 0.0)):
    r0, directions = source.generate_photons(energy, pos=source_pos)
    
    # расчет точек пересечения лучей со сферой радиуса R
    b = 2 * np.sum(r0 * directions, axis=1)
    c = np.sum(r0**2, axis=1) - detector.R**2
    t = (-b + np.sqrt(b**2 - 4 * c)) / 2.0
    
    hit_points = r0 + directions * t[:, np.newaxis]
    
    # считаем попадания по каналам ФЭУ
    pmt_hits = np.zeros(detector.N, dtype=int)
    for hit in hit_points:
        idx = detector.check_hit_pmt(hit)
        if idx != -1:
            pmt_hits[idx] += 1
            
    # отклик фотокатода (распределение пуассона со средним a * hits)
    pe_per_pmt = np.random.poisson(detector.a * pmt_hits)
    total_charge = np.sum(pe_per_pmt)
    
    return total_charge, pmt_hits

detector = Detector()
source = Source()
energy = 1.0  # 1 МэВ

total_charge, pmt_hits = simulate_single_event(detector, source, energy, source_pos=(0.0, 0.0, 0.0))

# теоретический расчет 
expected_geom_fraction = (detector.N * detector.r**2) / (4 * detector.R**2)
expected_mean_charge = detector.a * source.Y * energy * expected_geom_fraction

print(f"Суммарный собранный заряд (симуляция): {total_charge} фотоэлектронов")
print(f"Ожидаемый средний заряд по формуле: {expected_mean_charge:.2f} фотоэлектронов")

Суммарный собранный заряд (симуляция): 126 фотоэлектронов
Ожидаемый средний заряд по формуле: 125.00 фотоэлектронов
